# PE_SEOBNR_plot: Before/After Reweight Corner Plots

Showcase 4 events: 2 with extreme **q bias**, 2 with extreme **dL bias**.

Uses `bilby.core.result.plot_multiple` for overlaid corner plots, matching the style of `PE_eosfit_plot.ipynb`.

### Event selection
| Event | SNR | Type | Key feature |
|---|---|---|---|
| event_0054 | 109.1 | q bias | dq=−0.25, q_true=0.97 → q_PE=0.71 |
| event_0057 | 30.9 | q bias | dq=−0.23, **dL偏低** (独立于dL偏) |
| event_0058 | 31.1 | dL bias | dL偏2.3×, **q几乎无偏** |
| event_0051 | 34.3 | dL bias | dL偏1.9×, q_true=0.99 |

In [ ]:
import os, json
import numpy as np, pandas as pd
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
import bilby
from wcosmo.astropy import Planck18
from astropy.cosmology import FlatLambdaCDM

RUNDIR = '/scratch/gpfs/ANDREASB/fy6204/GW/Workspace'
os.chdir(RUNDIR)

PE_OUTDIR = 'outputs/outdir_population_run_SEOBNR'
POP_OUTDIR = 'outputs/outdir_population_SEOBNR'

# Load all injection truths
with open(f'{POP_OUTDIR}/accepted.jsonl') as f:
    all_meta = [json.loads(line) for line in f]

# EOS-fit helpers (from PE_eosfit_reweight.py)
H0_TRUE = float(Planck18.H0.value)
LAMBDA_FIT_NORM = 3500.0
A0_FIT, A1_FIT, A2_FIT = -0.30781804, 0.79244108, -0.51480556
cosmo_fid = FlatLambdaCDM(H0=70.0, Om0=float(Planck18.Om0))
_z = np.linspace(0.0, 2.0, 20000)
_dL_grid = np.asarray(cosmo_fid.luminosity_distance(_z).value, dtype=float)

def z_from_dL_H0(dL_mpc, H0):
    dL_scaled = np.asarray(dL_mpc,float)*(np.asarray(H0,float)/70.0)
    return np.interp(np.clip(dL_scaled,_dL_grid[0],_dL_grid[-1]),_dL_grid,_z)

def add_eosfit_derived(df):
    df = df.copy()
    dL = df["luminosity_distance"].to_numpy(dtype=float)
    H0 = df["H0_sample"].to_numpy(dtype=float) if "H0_sample" in df.columns else np.full(len(df), H0_TRUE)
    q = df["mass_ratio"].to_numpy(dtype=float)
    Mc = df["chirp_mass"].to_numpy(dtype=float)
    m1, m2 = bilby.gw.conversion.chirp_mass_and_mass_ratio_to_component_masses(Mc, q)
    z = z_from_dL_H0(dL, H0)
    m1s=m1/(1+z); m2s=m2/(1+z)
    da0=df["delta_a0"].to_numpy(dtype=float)
    da1=df["delta_a1"].to_numpy(dtype=float)
    da2=df["delta_a2"].to_numpy(dtype=float)
    def lam(mbar,d0,d1,d2):
        return np.maximum(LAMBDA_FIT_NORM*(1+A0_FIT*(1+d0)+A1_FIT*(1+d1)*mbar+A2_FIT*(1+d2)*mbar**2)/mbar**5, 1e-8)
    lam1=lam(m1s,da0,da1,da2); lam2=lam(m2s,da0,da1,da2)
    df["lambda_tilde"]=bilby.gw.conversion.lambda_1_lambda_2_to_lambda_tilde(lam1,lam2,m1,m2)
    df["delta_lambda_tilde"]=bilby.gw.conversion.lambda_1_lambda_2_to_delta_lambda_tilde(lam1,lam2,m1,m2)
    return df

# Label map
LABEL_MAP = {
    "mass_ratio": r"$q$", "chirp_mass": r"$\mathcal{M}_c$",
    "luminosity_distance": r"$d_L$", "H0_sample": r"$H_0$",
    "lambda_tilde": r"$\tilde{\Lambda}$", "delta_lambda_tilde": r"$\delta\tilde{\Lambda}$",
    "delta_a0": r"$\delta a_0$", "delta_a1": r"$\delta a_1$", "delta_a2": r"$\delta a_2$",
}

print("Imports OK")

In [ ]:
def make_result(df, params, label):
    priors = {}
    for p in params:
        x = df[p].dropna().to_numpy(dtype=float)
        lo, hi = np.percentile(x, [0.1, 99.9]) if len(x) > 0 else (0, 1)
        priors[p] = bilby.core.prior.Uniform(lo, hi, name=p)
    labels = [LABEL_MAP.get(p, p) for p in params]
    return bilby.core.result.Result(
        label=label, outdir='.', sampler='dynesty',
        search_parameter_keys=params, priors=priors,
        posterior=df[params].copy(),
        parameter_labels=labels, parameter_labels_with_unit=labels,
    )

def add_truth_lines(fig, params, truths):
    ndim = len(params)
    axes = np.array(fig.axes).reshape((ndim, ndim))
    for i, yp in enumerate(params):
        yt = truths.get(yp)
        if yt is not None and np.isfinite(yt):
            axes[i,i].axvline(yt, color='red', lw=1.5)
        for j in range(i):
            xp = params[j]; xt = truths.get(xp)
            if xt is not None and np.isfinite(xt):
                axes[i,j].axvline(xt, color='red', lw=1.0)
            if yt is not None and np.isfinite(yt):
                axes[i,j].axhline(yt, color='red', lw=1.0)
    return fig

def plot_event(ev_idx):
    ev_name = f"event_{ev_idx:04d}"
    tag = f"bns_{ev_name}_seobnr"
    meta = all_meta[ev_idx - 1]
    m1d = float(meta["mass_1_detector"]); m2d = float(meta["mass_2_detector"])
    snr = meta.get("network_snr", 0)
    dL_true = float(meta["luminosity_distance_mpc"])
    
    # Truths: including lambda_tilde, delta_lambda_tilde from injection
    lam1_inj = float(meta["lambda_1"]); lam2_inj = float(meta["lambda_2"])
    truths = {
        "mass_ratio": min(m1d,m2d)/max(m1d,m2d),
        "chirp_mass": (m1d*m2d)**(3./5.)/(m1d+m2d)**(1./5.),
        "luminosity_distance": dL_true,
        "H0_sample": H0_TRUE,
        "lambda_tilde": bilby.gw.conversion.lambda_1_lambda_2_to_lambda_tilde(lam1_inj, lam2_inj, m1d, m2d),
        "delta_lambda_tilde": bilby.gw.conversion.lambda_1_lambda_2_to_delta_lambda_tilde(lam1_inj, lam2_inj, m1d, m2d),
        "delta_a0": 0.0, "delta_a1": 0.0, "delta_a2": 0.0,
    }
    
    # Load before and after reweight
    before_path = f'{PE_OUTDIR}/{tag}_posterior_augmented.csv'
    rw_path = f'{PE_OUTDIR}/{tag}_reweighted_posterior_augmented.csv'
    
    df_before = add_eosfit_derived(pd.read_csv(before_path))
    df_after = add_eosfit_derived(pd.read_csv(rw_path))
    
    CORNER_PARAMS = [
        "mass_ratio", "chirp_mass", "luminosity_distance",
        "lambda_tilde", "delta_lambda_tilde",
        "H0_sample", "delta_a0", "delta_a1", "delta_a2",
    ]
    params = [p for p in CORNER_PARAMS if p in df_before.columns and p in df_after.columns]
    
    r_before = make_result(df_before, params, 'Before reweight')
    r_after = make_result(df_after, params, 'After reweight')
    
    # Determine bias type
    if ev_idx in [54, 57]: btype="q bias"
    else: btype="dL bias"
    
    print(f"{ev_name}: SNR={snr:.1f}, {btype}, {len(df_before)}->{len(df_after)} samples, {len(params)} params")
    
    fig = bilby.core.result.plot_multiple(
        [r_before, r_after],
        labels=['Before reweight', 'After reweight'],
        parameters=params, save=False, bins=30,
        plot_datapoints=False, smooth=0.9, quantiles=[0.16,0.84],
        colors=['C0','C3'],
    )
    add_truth_lines(fig, params, truths)
    
    save_path = f'fig_corner_{ev_name}_before_after_reweight.png'
    fig.savefig(save_path, dpi=200, bbox_inches='tight')
    plt.close(fig)
    print(f"  -> {save_path}")
    return fig

print("Functions defined OK")

## event_0054 — q bias (dq=−0.25, SNR=109)

In [ ]:
plot_event(54)

## event_0057 — q bias (dq=−0.23, dL偏低, SNR=31)

In [ ]:
plot_event(57)

## event_0058 — dL bias (dL ×2.3, q fine, SNR=31)

In [ ]:
plot_event(58)

## event_0051 — dL bias (dL ×1.9, q_true=0.99, SNR=34)

In [ ]:
plot_event(51)

## Summary

- **event_0054, 0057**: q systematically underestimated — posterior peaks far below truth red line
- **event_0058, 0051**: dL systematically overestimated, H0 pulled low — posterior in (dL, H0) far from truth
- Before/after reweight nearly identical — the bias is in the PE likelihood, not in the relative-binning reweighting